# Quickstart — Browserless × LangChain (Python)

Connect to the Browserless MCP server, list its tools, and run a ReAct agent that scrapes the web. Browserless ships 10 tools over a single streamable-HTTP MCP endpoint: 8 stateless (smartscraper, search, map, crawl, export, performance, function, download) plus a multi-turn browser agent (`browserless_agent` + `browserless_skill`).

Get a token at [account.browserless.io](https://account.browserless.io).

In [ ]:
%pip install -q langchain-mcp-adapters langgraph langchain-anthropic

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "browserless": {
        "transport": "http",
        "url": "https://mcp.browserless.io/mcp",
        "headers": {"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    }
})

tools = await client.get_tools()
print(f"Loaded {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}")

## Direct tool invocation

Each MCP tool is a regular LangChain `BaseTool`, so you can call it directly without an LLM in the loop. Here we use `browserless_smartscraper` to fetch a deterministic URL as markdown.

In [ ]:
smartscraper = next(t for t in tools if t.name == "browserless_smartscraper")
result = await smartscraper.ainvoke({
    "url": "https://example.com",
    "formats": ["markdown"],
})
print(result)

## ReAct agent (stateless tools)

Wrap the tools in a LangGraph ReAct agent. This example uses only the 8 stateless tools — every step is a single self-contained call, so it works regardless of session-id behavior.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic

stateless_tools = [t for t in tools if t.name not in ("browserless_agent", "browserless_skill")]

agent = create_react_agent(
    ChatAnthropic(model="claude-sonnet-4-6"),
    stateless_tools,
)

out = await agent.ainvoke({
    "messages": [{"role": "user", "content": "Scrape https://news.ycombinator.com and list the top 5 headlines as a markdown bullet list."}]
})
print(out["messages"][-1].content)